In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

**EDA**


In [2]:
df = pd.read_csv("restaurant_sales_data.csv")
df.head()

,Order ID,Customer ID,Category,Item,Price,Quantity,Order Total,Order Date,Payment Method
0,ORD_705844,CUST_092,Side Dishes,Side Salad,3.0,1.0,3.0,2023-12-21,Credit Card
1,ORD_338528,CUST_021,Side Dishes,Mashed Potatoes,4.0,3.0,12.0,2023-05-19,Digital Wallet
2,ORD_443849,CUST_029,Main Dishes,Grilled Chicken,15.0,4.0,60.0,2023-09-27,Credit Card
3,ORD_630508,CUST_075,Drinks,NaN,NaN,2.0,5.0,2022-08-09,Credit Card
4,ORD_648269,CUST_031,Main Dishes,Pasta Alfredo,12.0,4.0,48.0,2022-05-15,Cash


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17534 entries, 0 to 17533
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order ID        17534 non-null  str    
 1   Customer ID     17534 non-null  str    
 2   Category        17534 non-null  str    
 3   Item            15776 non-null  str    
 4   Price           16658 non-null  float64
 5   Quantity        17104 non-null  float64
 6   Order Total     17104 non-null  float64
 7   Order Date      17534 non-null  str    
 8   Payment Method  16452 non-null  str    
dtypes: float64(3), str(6)
memory usage: 1.2 MB


In [4]:
df.describe()

,Price,Quantity,Order Total
count,16658.000000,17104.000000,17104.000000
mean,6.586325,3.014149,19.914494
std,4.834652,1.414598,18.732549
min,1.000000,1.000000,1.000000
25%,3.000000,2.000000,7.500000
50%,5.000000,3.000000,15.000000
75%,7.000000,4.000000,25.000000
max,20.000000,5.000000,100.000000


In [5]:
print(f"Total de Duplicados: {df.duplicated().sum()}")


Total de Duplicados: 0


In [6]:
print("\033[1mTotal de Valores Nulos")
df.isnull().sum()

Total de Valores Nulos


Order ID             0
Customer ID          0
Category             0
Item              1758
Price              876
Quantity           430
Order Total        430
Order Date           0
Payment Method    1082
dtype: int64

In [7]:
# Cambiar el formato de fecha"
df['Order Date'] = pd.to_datetime(df['Order Date'],errors='coerce')
df['Price'] = pd.to_numeric(df['Price'],errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'],errors='coerce')
df['Order Total'] = pd.to_numeric(df['Order Total'],errors='coerce')

In [8]:
# Llenar el Payment Method 
df['Payment Method'] = df['Payment Method'].ffill()

In [9]:
# Llenar los valores númericos faltantes
dfOrderTotalNA = df['Order Total'].isna() & df['Price'].notna() & df['Quantity'].notna()
df.loc[dfOrderTotalNA,"Order Total"] = df.loc[dfOrderTotalNA,"Price"] * df.loc[dfOrderTotalNA,"Quantity"]
dfPriceNA = df['Price'].isna() & df['Order Total'].notna() & df['Quantity'].notna() & (df['Quantity'] != 0)
df.loc[dfPriceNA,"Price"] = df.loc[dfPriceNA,"Order Total"] / df.loc[dfPriceNA,"Quantity"]
dfQuantityNA = df['Quantity'].isna() & df['Price'].notna() & df['Order Total'].notna() & (df['Price'] != 0)
df.loc[dfQuantityNA,"Quantity"] = df.loc[dfQuantityNA,"Order Total"] / df.loc[dfQuantityNA,"Price"]
# Dropear los valores númericos faltantes
df = df.dropna(subset=['Order Total','Price','Quantity'], how='any')

In [10]:
print("\nModa calculada para cada grupo (categoría, precio):")
print(df.groupby(['Category', 'Price'])['Item'].apply(lambda x: x.mode()[0] if not x.mode().empty else None))


Moda calculada para cada grupo (categoría, precio):
Category     Price
Desserts     4.0             Fruit Salad
             5.0               Ice Cream
             6.0          Chocolate Cake
             7.0              Cheesecake
Drinks       1.0                   Water
             2.5               Coca Cola
             3.0            Orange Juice
Main Dishes  12.0          Pasta Alfredo
             14.0     Vegetarian Platter
             15.0        Grilled Chicken
             18.0                 Salmon
             20.0                  Steak
Side Dishes  3.0              Side Salad
             4.0         Mashed Potatoes
             5.0      Grilled Vegetables
Starters     4.0            French Fries
             5.0            Cheese Fries
             7.0              Beef Chili
             8.0            Chicken Melt
             10.0          Nachos Grande
Name: Item, dtype: str


In [11]:
# Llenar los items faltantes según precio y categoría
moda_items = df.groupby(['Category', 'Price'])['Item'].apply(lambda x: x.mode()[0] if not x.mode().empty else None)
for (categoria, precio), moda_item in moda_items.items():
        df.loc[(df['Category'] == categoria) & (df['Price'] == precio) & (df['Item'].isnull()), 'Item'] = moda_item

In [12]:
print("\033[1mTotal de Valores Nulos Despues de la Transformación")
df.isnull().sum()

Total de Valores Nulos Despues de la Transformación


Order ID          0
Customer ID       0
Category          0
Item              0
Price             0
Quantity          0
Order Total       0
Order Date        0
Payment Method    0
dtype: int64

In [13]:
df.to_csv('Cleaned Restaurant Sales.csv',index=False)